# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

We audited the FlyRank research paper and selected two findings to review constructively.

### Finding A: Finding #4 — The Freshness Multiplier (Page 9)
* **Claim:** The paper states that "365+ day content that was refreshed within 30 days shows a 3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039)."
* **Methodological Questions:**
  1. **Where does the label come from?** The "health score" is a composite index (Impressions + Position + CTR + Scroll depth). Comparing the average health score of refreshed vs. unrefreshed pages introduces selection bias. Editors selectively refresh pages that are already higher-performing or have high potential, so the "boost" is at least partially due to selection, not the refresh itself.
  2. **Does the validation design support the claim?** No, the validation is cross-sectional (comparing different sets of pages in a single snapshot) rather than longitudinal (comparing the same pages before and after the refresh). Moreover, the paper mentions that the `361+` day untouched bucket in the active sample has only 1 declining page, making the 283:1 growth ratio extremely unstable and susceptible to small-sample artifacts.

### Finding B: Finding #10 — AI Model Performance (Page 16)
* **Claim:** The paper states that "Within this mostly AI-authored portfolio, age-controlled model cohorts do not show a simple blanket penalty tied only to AI use. When we compared models inside the same age tiers, the differences moved around... Gemini leads some cohorts and OpenAI leads others."
* **Methodological Questions:**
  1. **Where does the label come from?** The label is average organic impressions or the composite health score.
  2. **Does the validation design support the claim?** While controlling for age is a step in the right direction, it doesn't control for topic/category or prompt/editing standards. If OpenAI models were deployed on high-intent transactional templates and Gemini on informational blog templates, the difference in traffic and position is driven by intent and template style, not LLM performance. The comparison lacks control for these confounding variables.

In [1]:
# Section 1 Confirmation
print("Audited Finding #4 (Freshness Multiplier) and Finding #10 (AI Model Performance) from the PDF paper.")

Audited Finding #4 (Freshness Multiplier) and Finding #10 (AI Model Performance) from the PDF paper.


## 2. My model under an honest split (before/after)

### Split Design Comparison:
* **Random Split (Before):** Standard row-based stratified split. It allows data leakage because multiple pages from the same client site are split between train and test. Since pages within the same client share domain authority, topical niche, and template structures, the model can memorize client-specific signals to make predictions.
* **Client Holdout Group Split (After):** We group observations by `client_id` and hold out 20% of clients entirely (6 out of 32) for testing. This tests how well the model generalizes to an entirely new website/domain.

Below, we train the Random Forest model under both split strategies and report the performance metrics next to the majority-class naive baseline rate.

In [2]:
# Re-run Week-5 Model under Random vs Grouped splits
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Load dataset
df = pd.read_csv('../../data/processed/refresh_feature_vector.csv')
target_series = df["is_declining_label"].astype(int)
base_rate = target_series.mean()

# Prepare feature matrices
MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

def build_feature_matrix(frame):
    numeric_features = [col for col in MODEL_NUMERIC_FEATURES if col in frame.columns]
    categorical_features = [col for col in MODEL_CATEGORICAL_FEATURES if col in frame.columns]
    
    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )
    
    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    return feature_frame, list(feature_frame.columns)

feature_frame, feature_columns = build_feature_matrix(df)

# --- 1. Random Split ---
train_idx_rand, test_idx_rand = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=target_series
)
train_X_rand = feature_frame.iloc[train_idx_rand]
test_X_rand = feature_frame.iloc[test_idx_rand]
train_y_rand = target_series.iloc[train_idx_rand]
test_y_rand = target_series.iloc[test_idx_rand]

rf_rand = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
)
rf_rand.fit(train_X_rand, train_y_rand)
probs_rand = rf_rand.predict_proba(test_X_rand)[:, 1]

# --- 2. Grouped Client Split ---
def make_client_aware_split(frame):
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()
    
    random_generator = np.random.default_rng(RANDOM_STATE)
    shuffled_clients = random_generator.permutation(unique_clients)
    test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
    test_clients = set(shuffled_clients[:test_client_count])
    
    test_mask = client_series.isin(test_clients).to_numpy()
    train_indices = all_indices[~test_mask]
    test_indices = all_indices[test_mask]
    
    return train_indices, test_indices, test_clients

train_idx_grp, test_idx_grp, test_clients = make_client_aware_split(df)
train_X_grp = feature_frame.iloc[train_idx_grp]
test_X_grp = feature_frame.iloc[test_idx_grp]
train_y_grp = target_series.iloc[train_idx_grp]
test_y_grp = target_series.iloc[test_idx_grp]

rf_grp = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
)
rf_grp.fit(train_X_grp, train_y_grp)
probs_grp = rf_grp.predict_proba(test_X_grp)[:, 1]

# --- Print Comparison Table ---
comparison_df = pd.DataFrame([
    {
        "Split Strategy": "Stratified Random Row Split (Before)",
        "Base Rate": f"{base_rate:.2%}",
        "Test Accuracy": f"{accuracy_score(test_y_rand, probs_rand >= 0.5):.2%}",
        "ROC AUC": f"{roc_auc_score(test_y_rand, probs_rand):.4f}",
        "Avg Precision": f"{average_precision_score(test_y_rand, probs_rand):.4f}"
    },
    {
        "Split Strategy": "Grouped Client Split (After)",
        "Base Rate": f"{base_rate:.2%}",
        "Test Accuracy": f"{accuracy_score(test_y_grp, probs_grp >= 0.5):.2%}",
        "ROC AUC": f"{roc_auc_score(test_y_grp, probs_grp):.4f}",
        "Avg Precision": f"{average_precision_score(test_y_grp, probs_grp):.4f}"
    }
])

print("=== Split Evaluation Before/After Comparison ===")
print(comparison_df.to_string(index=False))

# Quantify performance gap
roc_auc_gap = roc_auc_score(test_y_rand, probs_rand) - roc_auc_score(test_y_grp, probs_grp)
ap_gap = average_precision_score(test_y_rand, probs_rand) - average_precision_score(test_y_grp, probs_grp)
print(f"\nMemorization Gap - ROC AUC Drop: {roc_auc_gap:.4f}")
print(f"Memorization Gap - Avg Precision Drop: {ap_gap:.4f}")

=== Split Evaluation Before/After Comparison ===
                      Split Strategy Base Rate Test Accuracy ROC AUC Avg Precision
Stratified Random Row Split (Before)    54.21%        69.17%  0.7580        0.7687
        Grouped Client Split (After)    54.21%        67.14%  0.7474        0.6101

Memorization Gap - ROC AUC Drop: 0.0107
Memorization Gap - Avg Precision Drop: 0.1587


## 3. Leakage audit

### Leakage Taxonomy & Timeline Overlap:
1. **Label-derived / Sibling columns:** The target `is_declining_label` is derived from `trend_direction` which is computed from `trend_pct` (comparing the last 30 days of impressions with the 30 days before that). Features like `trend_pct` and `trend_direction` are strictly excluded from our modeling.
2. **Timeline Window Overlap:** This is the most critical issue. Our features contain trailing 90-day aggregates (e.g. `log_impressions_90d`, `log_clicks_90d`, `days_with_impressions`, `days_with_sessions`, etc.). The label window represents days 0-60 back. Since the 90-day feature aggregation period includes days 0-60 back, these features overlap with the label window. At a true moment of prediction (day 60 back), we would not have access to these future 90-day totals.
3. **Audit Strategy:** To test if the model is relying heavily on this timeline overlap leakage, we drop the top historical presence features (`log_impressions_90d` and `days_with_impressions`) and evaluate. If the metric collapses, it confesses to relying on look-ahead information.

In [3]:
# Run Leakage Audit
print("=== Feature Leakage Audit ===")

# Identify clean columns (excluding log_impressions_90d and days_with_impressions)
suspects = ["log_impressions_90d", "days_with_impressions"]
clean_columns = [col for col in train_X_grp.columns if not any(sus in col for sus in suspects)]

rf_clean = RandomForestClassifier(
    class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
)
rf_clean.fit(train_X_grp[clean_columns], train_y_grp)
probs_clean = rf_clean.predict_proba(test_X_grp[clean_columns])[:, 1]

print(f"Grouped Client split (All features) - ROC AUC: {roc_auc_score(test_y_grp, probs_grp):.4f}, Avg Precision: {average_precision_score(test_y_grp, probs_grp):.4f}")
print(f"Grouped Client split (No suspects)    - ROC AUC: {roc_auc_score(test_y_grp, probs_clean):.4f}, Avg Precision: {average_precision_score(test_y_grp, probs_clean):.4f}")
print(f"Performance Loss - ROC AUC Drop: {roc_auc_score(test_y_grp, probs_grp) - roc_auc_score(test_y_grp, probs_clean):.4f}")
print(f"Performance Loss - Avg Precision Drop: {average_precision_score(test_y_grp, probs_grp) - average_precision_score(test_y_grp, probs_clean):.4f}")

=== Feature Leakage Audit ===


Grouped Client split (All features) - ROC AUC: 0.7474, Avg Precision: 0.6101
Grouped Client split (No suspects)    - ROC AUC: 0.7138, Avg Precision: 0.5668
Performance Loss - ROC AUC Drop: 0.0336
Performance Loss - Avg Precision Drop: 0.0433


## 4. Claim rewrite

### Claim Rewrite:
* **Unsafe/Overconfident Claim:** *"The Random Forest model achieves high accuracy and proves that updating older thin pages will increase their search engine traffic and prevent decline."*
* **Why it is unsafe:** It claims causal direction ("will increase", "proves") from cross-sectional data, doesn't mention the base rate, and ignores the fact that the data has overlap window leakage.
* **Safe/Honest Claim:** *"In this portfolio dataset, we measured that content update frequency and historical impressions are associated with trend direction. Under a client-holdout validation design, the Random Forest model flags potential decline candidates with a precision of 61.0% on unseen domains (compared to a majority baseline rate of 54.2%). These predictions should be treated as decision-support indicators to prioritize editorial review, rather than causal guarantees of traffic growth."*

Below, we inspect prediction errors on the held-out test set to understand the model's limitations.

In [4]:
# Extract False Positives and False Negatives from the held-out test set
test_df_slice = df.iloc[test_idx_grp].copy()
test_df_slice["rf_prob"] = probs_grp
test_df_slice["rf_pred"] = (probs_grp >= 0.5).astype(int)

print("=== False Positive Analysis (Predicted Decline, Actual Stable/Up) ===")
# Sort by highest predicted probability of decline among actual negatives
fps = test_df_slice[(test_df_slice["rf_pred"] == 1) & (test_df_slice["is_declining_label"] == 0)].sort_values("rf_prob", ascending=False)
print(fps[['content_id', 'client_id', 'rf_prob', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'trend_direction', 'content_age_days']].head(3).to_string(index=False))

print("\n=== False Negative Analysis (Predicted Stable, Actual Declined) ===")
# Sort by lowest predicted probability of decline among actual positives
fns = test_df_slice[(test_df_slice["rf_pred"] == 0) & (test_df_slice["is_declining_label"] == 1)].sort_values("rf_prob", ascending=True)
print(fns[['content_id', 'client_id', 'rf_prob', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'trend_direction', 'content_age_days']].head(3).to_string(index=False))

print("\nObservation:")
print("1. False Positives reveal that pages with typical decline indicators (low CTR, newer page) sometimes remain stable or grow due to seasonal/external trends.")
print("2. False Negatives show that low-volume pages (e.g. impressions = 1 or 3) are labeled as 'declined' if they drop to 0. The model correctly ignores these pages because their traffic is virtually zero (low signal-to-noise ratio), but they count as validation errors. This highlights the need for a minimum visibility threshold in deployment.")

=== False Positive Analysis (Predicted Decline, Actual Stable/Up) ===
          content_id         client_id  rf_prob  impressions_90d  avg_position  ctr  days_since_last_update trend_direction  content_age_days
content_d2dffcc697a4 client_f74efabef1 0.737431             5091          14.1 0.20                      20          stable               144
content_00603b0349b4 client_f74efabef1 0.735212             1076          25.6 0.09                      20              up               125
content_e55b8ab078b0 client_f74efabef1 0.733797              369          21.8 0.00                      20          stable               112

=== False Negative Analysis (Predicted Stable, Actual Declined) ===
          content_id         client_id  rf_prob  impressions_90d  avg_position  ctr  days_since_last_update trend_direction  content_age_days
content_28b4223f4e5f client_98a3ab7c34 0.081668                1           0.0  0.0                       1            down                91
content_3

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.